In [1]:
%load_ext autoreload
%autoreload 2

# Quantile regression with tuned CatBoost

**Objective:** test whether direct conditional-quantile prediction gives useful point accuracy and better calibrated uncertainty for total alkalinity (`talk`).

The experiment uses the same preprocessing, nested spatial/temporal cross-validation, residualized linear baseline, and outer held-out split as `02_train_RMSE_CRPS_tuned.ipynb`. CatBoost predicts quantiles from 1% to 99% in one model. Hyperparameters are tuned directly on CatBoost's native `MultiQuantile` validation loss; the outer test set remains untouched until final evaluation.

In [2]:
import pathlib

import dotenv
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from sklearn import metrics

import highres_ta as ta

BASE = pathlib.Path(dotenv.find_dotenv("pyproject.toml")).parent

## Data loading and preprocessing

In [3]:
COORD_COLUMNS = [
    "expocode",
    "time",
    "lat",
    "lon",
    "depth",
]
TARGET_NAME = "talk"
FEATURE_NAMES = [
    "salinity",
    "temperature",
    "ssh_adt",
    "nitrate",
    "phosphate",
    "silicate",
    "bottomdepth",
]
QC_COLS = ["talkqc", "talkf"]
REQUIRED_COLUMNS = list(set(COORD_COLUMNS + FEATURE_NAMES + [TARGET_NAME] + QC_COLS))
NONAN_SUBSET = [TARGET_NAME, "salinity", "temperature", "nitrate", "ssh_adt"]
LINEAR_FEATURES = ["salinity", "temperature"]
FEATURE_NAMES += [
    "ncoord_x",
    "ncoord_y",
    "ncoord_z",
]

In [4]:
data = (
    ta.load_data()[REQUIRED_COLUMNS]
    .set_index(COORD_COLUMNS, drop=False)
    .pipe(ta.drop_extreme_salinities, min=20, max=40)
    .pipe(
        ta.add_talk_adjustment,
        fname=str(BASE / "data" / "glodapv2_adjustments_last_updated_on_2026_07_09.csv"),
    )
    .pipe(ta.drop_bad_quality_talk)
    .dropna(subset=NONAN_SUBSET)
    .drop_duplicates(subset=COORD_COLUMNS, keep="first")
    .select_dtypes(include=[np.number])
    .pipe(ta.add_cyclical_dayofyear)
    .pipe(ta.add_spherical_coords)
    .loc[:, FEATURE_NAMES + [TARGET_NAME]]
)

data.shape

2026-08-18 09:22:26.868 | DEBUG    | highres_ta.dataio:load_data:18 - Loading 40 .pq files from ../data/training
2026-08-18 09:22:27.154 | DEBUG    | highres_ta.target_filtering:drop_bad_quality_talk:49 - TA values with large adjustments (<= 6.0 mol/kg): 2365
2026-08-18 09:22:27.155 | DEBUG    | highres_ta.target_filtering:drop_bad_quality_talk:52 - TA values without good flags (!= 2): 3046
2026-08-18 09:22:27.155 | INFO     | highres_ta.target_filtering:drop_bad_quality_talk:53 - Number of rows filtered due to large adjustments and bad flags: 5319 of 41534 (13%)


(27210, 11)

## Outer train/test split

In [5]:
train_idx, test_idx = ta.make_train_test_folds(data, n_splits=6)[5]
test = data.iloc[test_idx]
train = data.iloc[train_idx]
train_folds = ta.make_train_test_folds(train, shuffle=False)

pd.Series({"train": len(train), "test": len(test)}, name="observations")

2026-08-18 09:03:25.467 | DEBUG    | highres_ta.train_test_split:make_salinity_bins:48 - Using the following bin edges for salinity: [20.1909  32.8038  34.05706 34.819   35.535   39.231  ]
2026-08-18 09:03:25.469 | DEBUG    | highres_ta.train_test_split:stratified_group_folds:96 - Making train-test splits stratified by salinity_bin and grouped by expocode
2026-08-18 09:03:25.543 | DEBUG    | highres_ta.train_test_split:make_salinity_bins:48 - Using the following bin edges for salinity: [20.1909 32.8036 34.0576 34.819  35.535  39.231 ]
2026-08-18 09:03:25.545 | DEBUG    | highres_ta.train_test_split:stratified_group_folds:96 - Making train-test splits stratified by salinity_bin and grouped by expocode


train    22675
test      4535
Name: observations, dtype: int64

## Experiment plan

- Fit all requested quantiles jointly with CatBoost's `MultiQuantile` objective.
- Within each inner fold, fit the linear baseline only on that fold's training observations, preventing leakage.
- Tune the hyperparameters by minimizing the median native `MultiQuantile` validation loss across folds.
- Keep median RMSE, approximate CRPS, calibration, and crossing rates as diagnostics rather than optimization objectives.
- Refit once on the full outer-training set and evaluate point accuracy, calibration, interval coverage, sharpness, and quantile crossing on the held-out test set.

The selected grid includes 1%, 2.5%, 97.5%, and 99% tail quantiles while retaining conventional 50%, 80%, 90%, and 95% prediction intervals.

## Native MultiQuantile-loss hyperparameter tuning

In [ ]:
RANDOM_SEED = 42
N_TRIALS = 400
N_JOBS = -1
NUM_THREADS = 1
EARLY_STOPPING_ROUNDS = 50

QUANTILES = np.unique(
    np.round(
        np.r_[0.01, 0.025, 0.05, 0.1, np.arange(0.25, 1.0, 0.25), 0.9, 0.95, 0.975, 0.99],
        decimals=3,
    )
)
MEDIAN_INDEX = int(np.flatnonzero(np.isclose(QUANTILES, 0.5))[0])
ALPHA_STRING = ",".join(f"{alpha:g}" for alpha in QUANTILES)
ALWAYS_IGNORED_FEATURES = sorted(set(COORD_COLUMNS).intersection(FEATURE_NAMES))

fixed_catboost_params = {
    "loss_function": f"MultiQuantile:alpha={ALPHA_STRING}",
    "iterations": 2000,
    "random_seed": RANDOM_SEED,
    "ignored_features": ALWAYS_IGNORED_FEATURES,
    "allow_writing_files": False,
    "thread_count": NUM_THREADS,
    "verbose": False,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
}

pd.Series(
    {
        "number_of_quantiles": len(QUANTILES),
        "lowest_quantile": QUANTILES.min(),
        "highest_quantile": QUANTILES.max(),
        "median_column": MEDIAN_INDEX,
    },
    name="configuration",
)

### Quantile prediction and scoring helpers

`CatBoostResidualRegressor` fits a linear baseline and then models its residuals. For a multi-output quantile loss, each residual quantile must be shifted by the same linear prediction; `predict_quantiles` performs that combination explicitly.

For quantile function \(Q(\alpha)\), CRPS equals twice the pinball loss integrated over \(\alpha\). The finite grid below therefore gives a close numerical approximation, but it is not exactly the same calculation as the closed-form Gaussian CRPS in notebook 02.

In [7]:
def predict_quantiles(model, X):
    'Return absolute-target quantiles from a residualized MultiQuantile model.'
    X_frame = model._validate_feature_frame(X)
    linear_prediction = np.asarray(model.linear_model_.predict(X_frame), dtype=float)
    residual_quantiles = np.asarray(model.boosting_model_.predict(X_frame), dtype=float)

    if residual_quantiles.ndim != 2 or residual_quantiles.shape[1] != len(QUANTILES):
        raise ValueError(
            f"Expected {len(QUANTILES)} quantile columns, got {residual_quantiles.shape}"
        )
    return linear_prediction[:, None] + residual_quantiles


def pinball_loss_by_quantile(y_true, prediction):
    'Return mean pinball loss at each configured quantile.'
    y_true = np.asarray(y_true, dtype=float).reshape(-1, 1)
    prediction = np.asarray(prediction, dtype=float)
    error = y_true - prediction
    return np.mean(np.maximum(QUANTILES * error, (QUANTILES - 1.0) * error), axis=0)


def quantile_crps(y_true, prediction):
    'Approximate mean CRPS as twice integrated mean pinball loss.'
    return float(2.0 * np.trapezoid(pinball_loss_by_quantile(y_true, prediction), QUANTILES))


def quantile_calibration(y_true, prediction):
    'Return empirical P(Y <= predicted quantile) for every alpha.'
    y_true = np.asarray(y_true, dtype=float).reshape(-1, 1)
    return np.mean(y_true <= np.asarray(prediction, dtype=float), axis=0)


def quantile_column(alpha):
    matches = np.flatnonzero(np.isclose(QUANTILES, alpha))
    if len(matches) != 1:
        raise KeyError(f"Quantile {alpha:g} is not uniquely present in QUANTILES")
    return int(matches[0])

In [8]:
def objective(trial: optuna.Trial) -> float:
    trial_params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 100.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
        "depth": trial.suggest_int("depth", 4, 12),
        "rsm": trial.suggest_float("rsm", 0.1, 1.0),
        "random_strength": trial.suggest_float("random_strength", 1.0, 10.0),
    }

    fold_multi_quantile_loss = []
    fold_rmse = []
    fold_crps = []
    fold_calibration_mae = []
    fold_crossing_rate = []
    fold_best_iterations = []
    boost_kwargs = fixed_catboost_params | trial_params
    loss_name = fixed_catboost_params["loss_function"]

    for fold_train_idx, fold_valid_idx in train_folds:
        fold_model = ta.CatBoostResidualRegressor(
            linear_features=LINEAR_FEATURES,
            feature_names=FEATURE_NAMES,
            **boost_kwargs,
        )

        x_train = train.iloc[fold_train_idx].loc[:, FEATURE_NAMES]
        y_train = train.iloc[fold_train_idx].loc[:, TARGET_NAME]
        x_valid = train.iloc[fold_valid_idx].loc[:, FEATURE_NAMES]
        y_valid = train.iloc[fold_valid_idx].loc[:, TARGET_NAME]

        fold_model.fit(x_train, y_train, eval_set=(x_valid, y_valid))
        prediction = predict_quantiles(fold_model, x_valid)
        calibration = quantile_calibration(y_valid, prediction)
        best_validation_scores = fold_model.boosting_model_.get_best_score()["validation"]

        fold_multi_quantile_loss.append(float(best_validation_scores[loss_name]))
        fold_rmse.append(
            float(metrics.root_mean_squared_error(y_valid, prediction[:, MEDIAN_INDEX]))
        )
        fold_crps.append(quantile_crps(y_valid, prediction))
        fold_calibration_mae.append(float(np.mean(np.abs(calibration - QUANTILES))))
        fold_crossing_rate.append(
            float(np.mean(np.any(np.diff(prediction, axis=1) < 0, axis=1)))
        )
        fold_best_iterations.append(fold_model.boosting_model_.get_best_iteration() + 1)

    trial.set_user_attr("fold_multi_quantile_loss", fold_multi_quantile_loss)
    trial.set_user_attr("fold_rmse", fold_rmse)
    trial.set_user_attr("fold_crps", fold_crps)
    trial.set_user_attr("fold_calibration_mae", fold_calibration_mae)
    trial.set_user_attr("fold_crossing_rate", fold_crossing_rate)
    trial.set_user_attr("fold_best_iterations", fold_best_iterations)
    trial.set_user_attr("refit_iterations", int(np.median(fold_best_iterations)))

    return float(np.median(fold_multi_quantile_loss))

In [ ]:
sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    storage="sqlite:///catboost_quantile_cv.db",
    study_name="catboost_multi_quantile_native_loss_v03",
    load_if_exists=True,
)
study.set_metric_names(["multi_quantile_loss"])
study.optimize(objective, n_trials=N_TRIALS, n_jobs=N_JOBS, show_progress_bar=True)

[I 2026-08-18 09:03:33,541] A new study created in RDB with name: catboost_multi_quantile_native_loss_v03
/var/folders/4n/tkh2q3cn5fl09xv_kq_17v8c0000gn/T/ipykernel_72111/15160003.py:9: ExperimentalWarning: optuna.study.study.Study.set_metric_names is experimental (supported from v3.2.0). The interface can change in the future.
  study.set_metric_names(["multi_quantile_loss"])


  0%|          | 0/400 [00:00<?, ?it/s]

[I 2026-08-18 09:04:26,082] Trial 1 finished with value: {'multi_quantile_loss': 2.8271211684912654} and parameters: {'learning_rate': 0.2380035889015835, 'l2_leaf_reg': 3.0700175608798848, 'min_data_in_leaf': 45, 'depth': 11, 'rsm': 0.3933205738902146, 'random_strength': 7.72121933075333}. Best is trial 1 with value: 2.8271211684912654.
[I 2026-08-18 09:04:47,347] Trial 4 finished with value: {'multi_quantile_loss': 2.735802777403989} and parameters: {'learning_rate': 0.259922636084093, 'l2_leaf_reg': 10.418291214058767, 'min_data_in_leaf': 58, 'depth': 8, 'rsm': 0.737087933829821, 'random_strength': 2.762455028526871}. Best is trial 4 with value: 2.735802777403989.


## Best-trial selection

The study has one objective: the median of CatBoost's best native `MultiQuantile` validation loss across the inner folds. The lowest-loss completed trial is selected for the final refit.

In [ ]:
from optuna import visualization

cv_results = study.trials_dataframe(
    attrs=("number", "value", "params", "user_attrs", "state")
)
selected_trial = study.best_trial

print(f"Selected trial: {selected_trial.number}")
print(f"Selected median CV MultiQuantile loss: {selected_trial.value:.3f}")
print(f"Median diagnostic CV RMSE: {np.median(selected_trial.user_attrs['fold_rmse']):.3f}")
print(f"Median diagnostic CV quantile CRPS: {np.median(selected_trial.user_attrs['fold_crps']):.3f}")
print(f"Refit iterations: {selected_trial.user_attrs['refit_iterations']}")

display(cv_results.sort_values("value").head(20).reset_index(drop=True))
visualization.plot_optimization_history(study, target_name="CV MultiQuantile loss")

# Refit and held-out evaluation

The selected hyperparameters are refit on every outer-training observation. All metrics below use the held-out outer test split, which was not used by Optuna, early stopping, or trial selection.

In [ ]:
best_params = {
    **(fixed_catboost_params | selected_trial.params),
    "iterations": selected_trial.user_attrs["refit_iterations"],
}
print(best_params)

boosted_trees_model = ta.CatBoostResidualRegressor(
    LINEAR_FEATURES,
    FEATURE_NAMES,
    polynomial_degree=1,
    **best_params,
)
boosted_trees_model.fit(train[FEATURE_NAMES], train[TARGET_NAME])

feature_importance = pd.Series(
    boosted_trees_model.boosting_model_.feature_importances_,
    index=boosted_trees_model.boosting_model_.feature_names_,
).sort_values()
feature_importance.plot.barh(figsize=(5, 4), title="Feature importance")

In [ ]:
subset = test.loc[test["bottomdepth"] > 300].copy()
test_prediction = predict_quantiles(boosted_trees_model, subset[FEATURE_NAMES])
test_median = test_prediction[:, MEDIAN_INDEX]
test_residual = test_median - subset[TARGET_NAME].to_numpy()
test_calibration = quantile_calibration(subset[TARGET_NAME], test_prediction)
test_pinball = pinball_loss_by_quantile(subset[TARGET_NAME], test_prediction)

test_metrics = pd.Series(
    {
        "count": len(subset),
        "median_rmse": metrics.root_mean_squared_error(subset[TARGET_NAME], test_median),
        "quantile_crps": quantile_crps(subset[TARGET_NAME], test_prediction),
        "mean_pinball_loss": np.mean(test_pinball),
        "median_mae": metrics.mean_absolute_error(subset[TARGET_NAME], test_median),
        "median_bias_mean": np.mean(test_residual),
        "median_bias_median": np.median(test_residual),
        "median_r2": metrics.r2_score(subset[TARGET_NAME], test_median),
        "calibration_mae": np.mean(np.abs(test_calibration - QUANTILES)),
        "quantile_crossing_rate": np.mean(np.any(np.diff(test_prediction, axis=1) < 0, axis=1)),
    },
    name="held_out_test",
)
test_metrics.round(4)

## Calibration and interval sharpness

In [ ]:
calibration_table = pd.DataFrame(
    {
        "nominal_quantile": QUANTILES,
        "empirical_fraction_below": test_calibration,
        "calibration_error": test_calibration - QUANTILES,
        "pinball_loss": test_pinball,
    }
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(QUANTILES, test_calibration, marker="o", markersize=3)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
axes[0].set(
    xlabel="Nominal quantile",
    ylabel="Empirical fraction below prediction",
    title="Held-out quantile calibration",
    xlim=(0, 1),
    ylim=(0, 1),
)
axes[1].plot(QUANTILES, test_pinball, marker="o", markersize=3)
axes[1].set(
    xlabel="Quantile",
    ylabel="Mean pinball loss",
    title="Held-out pinball loss",
)
fig.tight_layout()

calibration_table.round(4)

In [ ]:
intervals = {
    "50%": (0.25, 0.75),
    "80%": (0.10, 0.90),
    "90%": (0.05, 0.95),
    "95%": (0.025, 0.975),
}
interval_rows = []
y_test = subset[TARGET_NAME].to_numpy()

for label, (lower_alpha, upper_alpha) in intervals.items():
    lower = test_prediction[:, quantile_column(lower_alpha)]
    upper = test_prediction[:, quantile_column(upper_alpha)]
    valid_order = lower <= upper
    interval_rows.append(
        {
            "interval": label,
            "nominal_coverage": upper_alpha - lower_alpha,
            "empirical_coverage": np.mean((y_test >= lower) & (y_test <= upper)),
            "mean_width": np.mean(upper - lower),
            "median_width": np.median(upper - lower),
            "ordered_fraction": np.mean(valid_order),
        }
    )

interval_results = pd.DataFrame(interval_rows).set_index("interval")
interval_results.round(4)

## Spatial pattern of median residuals

In [ ]:
coords = subset.index.to_frame()
coords["lat025"] = (coords["lat"] * 4) // 4 + 0.125
coords["lon025"] = (coords["lon"] * 4) // 4 + 0.125
grid_index = pd.MultiIndex.from_arrays(
    [coords["lat025"].to_numpy(), coords["lon025"].to_numpy()],
    names=["lat", "lon"],
)

(
    pd.Series(test_residual, index=grid_index)
    .groupby(["lat", "lon"])
    .mean()
    .to_xarray()
    .reindex(
        lon=np.arange(-180 + 0.125, 180, 0.25),
        lat=np.arange(-90 + 0.125, 90, 0.25),
    )
    .coarsen(lat=8, lon=8)
    .mean()
    .plot.imshow(
        center=0,
        vmin=-30,
        vmax=30,
        cmap="RdBu_r",
        aspect=2,
        size=6,
    )
)

## Interpretation checklist and next steps

- Use the native `MultiQuantile` CV loss to compare tuning trials; it is the loss the gradient booster optimized directly.
- Compare held-out median RMSE and bias with notebook 02's predictive mean.
- Prefer lower held-out CRPS/pinball loss only when calibration and interval widths are also credible; very wide intervals can cover well without being useful.
- Inspect `quantile_crossing_rate` and `ordered_fraction`. Material crossing suggests adding post-processing or fitting constrained/non-crossing quantiles.
- Treat the finite-grid `quantile_crps` as an approximation when comparing it with notebook 02's exact Gaussian CRPS.
- If performance is promising, persist the fitted model together with `QUANTILES`, since column order is part of the prediction contract.